# Etapa 4: Almacenamiento con Delta Lake
En este notebook se leen nuevamente las tablas guardadas en formato Delta Lake desde el notebook 01 y se valida su integridad.

In [0]:
# Apuntar al catálogo y schema del trabajo
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA tp7_peliculas")

# Leer nuevamente desde Delta Lake
df_delta = spark.table("peliculas_ratings")
top_delta = spark.table("top_peliculas")

df_delta.show(5)
df_delta.printSchema()

+-------+------+------+---------+----------+-----------+----------+--------------------+--------------------+----+
|movieId|userId|rating|timestamp|     fecha|anio_rating|valoracion|               title|              genres|anio|
+-------+------+------+---------+----------+-----------+----------+--------------------+--------------------+----+
|    349|     1|   4.0|964982563|2000-07-30|       2000|      Alta|Clear and Present...|Action|Crime|Dram...|1994|
|    593|     1|   4.0|964983793|2000-07-30|       2000|      Alta|Silence of the La...|Crime|Horror|Thri...|1991|
|   1552|     1|   4.0|964982620|2000-07-30|       2000|      Alta|      Con Air (1997)|Action|Adventure|...|1997|
|   2616|     1|   4.0|964983080|2000-07-30|       2000|      Alta|   Dick Tracy (1990)|        Action|Crime|1990|
|   2716|     1|   5.0|964983414|2000-07-30|       2000|      Alta|Ghostbusters (a.k...|Action|Comedy|Sci-Fi|1984|
+-------+------+------+---------+----------+-----------+----------+-------------

In [0]:
from pyspark.sql import functions as F

# 1) Conteo de filas
print("Filas en peliculas_ratings:", df_delta.count())
print("Filas en top_peliculas:", top_delta.count())

# 2) Nulos en columnas clave
df_delta.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c)
     for c in ["userId", "movieId", "rating", "title", "anio"]]
).show()

# 3) Duplicados (tiene que dar 0)
total = df_delta.count()
distintos = df_delta.dropDuplicates().count()
print("Duplicados:", total - distintos)

# 4) Rango de puntuaciones (tiene que ir de 0.5 a 5.0)
df_delta.select(F.min("rating").alias("min"), F.max("rating").alias("max")).show()

Filas en peliculas_ratings: 100818
Filas en top_peliculas: 450
+------+-------+------+-----+----+
|userId|movieId|rating|title|anio|
+------+-------+------+-----+----+
|     0|      0|     0|    0|   0|
+------+-------+------+-----+----+

Duplicados: 0
+---+---+
|min|max|
+---+---+
|0.5|5.0|
+---+---+



In [0]:
# Muestra el historial de versiones de la tabla Delta
spark.sql("DESCRIBE HISTORY workspace.tp7_peliculas.peliculas_ratings").show(truncate=False)

+-------+-------------------+--------------+-------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------------------------------------------------+
|version|timestamp          |userId        |userName           |operation                        |operationParameters                                                                                                                       